# Brain-to-Text Metrics v2

This notebook evaluates NeuroVLM brain-to-text generation separately from text-to-brain. It keeps the semantic metrics from `new_neuropvlm_metrics_v2`, removes the cleaned abstract branch, adds Recall@1 retrieval, and adds network label accuracy for the Networks test set.

This notebook is configured to run the full available networks, PubMed test, and NeuroVault evaluation sets by default. For fast iteration, set `MAX_B2T` to a small integer.

In [28]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from neurovlm import NeuroVLM
from neurovlm.data import load_dataset, load_latent
from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer, util as st_util
from transformers import AutoModel, AutoTokenizer


In [29]:
LLM_BACKEND = "huggingface"
LLM_MODEL = "qwen2.5:7b-instruct"
BERTSCORE_MODEL = "microsoft/deberta-xlarge-mnli"

MAX_B2T = None        # full available networks, PubMed test, and NeuroVault sets
RUN_NETWORKS = True
RUN_PUBMED = True
RUN_NEUROVAULT = True
RUN_GENERATED_TEXT = False  # slow LLM/BERTScore generated-text evaluation

B2T_TOP_K = 5
B2T_SIM_THR = 0.35
B2T_DATASETS = ["llm_neuro_terms", "kg_mesh", "cogatlas"]
OUTPUT_DIR = Path("docs/03_evaluation/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [30]:
nvlm = NeuroVLM()
_st_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Ready.")

Ready.


In [31]:
def load_network_test_set_labels_table():
    try:
        return load_dataset("network_test_set_labels"), "huggingface:neurovlm/embedded_text/network_test_set_labels.csv"
    except Exception as hf_error:
        candidates = [
            Path("network_test_set_labels.csv"),
            Path("docs/03_evaluation/network_test_set_labels.csv"),
            Path.cwd() / "network_test_set_labels.csv",
            Path.cwd() / "docs/03_evaluation/network_test_set_labels.csv",
        ]
        for candidate in candidates:
            if candidate.exists():
                return pd.read_csv(candidate), str(candidate.resolve())
        raise FileNotFoundError(
            "Could not load network_test_set_labels.csv from Hugging Face or local paths. "
            "Upload it to neurovlm/embedded_text as network_test_set_labels.csv. "
            f"Hugging Face error: {hf_error}"
        )


network_labels_df, NETWORK_TEST_SET_SOURCE = load_network_test_set_labels_table()
network_labels_df = network_labels_df[network_labels_df["network_key"] != "unknown"].copy()

# One row per canonical network, derived from the network test-set CSV.
network_info = (
    network_labels_df
    .sort_values(["network_key", "raw_network_label"])
    .groupby("network_key", as_index=False)
    .agg(
        display=("network_name", "first"),
        short_definition=("short_definition", "first"),
        long_definition=("long_definition", "first"),
        mapped_terms=("mapped_terms", "first"),
        raw_aliases=("raw_network_label", lambda s: "; ".join(sorted(set(map(str, s))))),
    )
)

DISPLAY_TO_KEY = dict(zip(network_info["display"], network_info["network_key"]))
KEY_TO_DISPLAY = dict(zip(network_info["network_key"], network_info["display"]))
SHORT_LABELS = dict(zip(network_info["display"], network_info["short_definition"]))
LONG_LABELS = dict(zip(network_info["display"], network_info["long_definition"]))

print(f"Loaded {len(network_info)} canonical network labels from {NETWORK_TEST_SET_SOURCE}")
display(network_info[["network_key", "display", "short_definition"]])

Loaded 8 canonical network labels from huggingface:neurovlm/embedded_text/network_test_set_labels.csv


,network_key,display,short_definition
0,attention,Attention,Dorsal attention network for selective attenti...
1,auditory,Auditory,"Auditory network for auditory perception, soun..."
2,cingulo_opercular,Cingulo-Opercular,Cingulo-opercular/salience network for salienc...
3,default_mode,Default Mode,Default mode network for self-referential thou...
4,frontoparietal_control,Frontoparietal Control,Frontoparietal control network for executive c...
5,language,Language,"Language network for speech comprehension, spe..."
6,motor,Motor,"Sensorimotor network for movement planning, vo..."
7,visual,Visual,"Visual network for visual perception, object r..."


In [32]:
def _normalize_expected_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "")).strip()

all_net_latents = load_latent("networks_neuro")
network_label_rows = (
    network_labels_df
    .drop_duplicates("raw_network_label")
    .set_index("raw_network_label")
)

networks_data = {}
for atlas_name, atlas_latents in all_net_latents.items():
    if not hasattr(atlas_latents, "items"):
        continue
    for raw_label, latent in atlas_latents.items():
        if raw_label not in network_label_rows.index:
            continue
        row = network_label_rows.loc[raw_label]
        sample_name = f"{atlas_name}:{raw_label}"
        networks_data[sample_name] = {
            "latent": latent,
            "short_gt": _normalize_expected_text(row["short_definition"]),
            "long_gt": _normalize_expected_text(row["long_definition"]),
            "network_key": row["network_key"],
            "display": row["network_name"],
            "atlas": atlas_name,
            "raw_network_label": raw_label,
        }

print(f"Networks loaded: {len(networks_data)} labeled maps across {len(all_net_latents)} atlases")
display(pd.DataFrame([
    {"sample": name, "network_key": d["network_key"], "display": d["display"]}
    for name, d in networks_data.items()
]).head())

Networks loaded: 119 labeled maps across 10 atlases


,sample,network_key,display
0,Du:CG-OP,cingulo_opercular,Cingulo-Opercular
1,Du:DN-B,default_mode,Default Mode
2,Du:SMOT-B,motor,Motor
3,Du:AUD,auditory,Auditory
4,Du:PM-PPr,motor,Motor


In [33]:
# PubMed summaries dataset: use summary text as long-form ground truth.
# These summaries better match the desired B2T LLM output than full abstracts.
df_summaries = load_dataset("pubmed_summaries")
df_pubs = load_dataset("pubmed_text")

if "test" not in df_summaries.columns:
    raise ValueError("pubmed_summaries must include a boolean 'test' column for PubMed evaluation.")
df_test = df_summaries[df_summaries["test"].fillna(False).astype(bool)].reset_index(drop=True)

_pmid_col = "pmid" if "pmid" in df_test.columns else df_test.columns[0]
_summary_col = "summary" if "summary" in df_test.columns else "description"
_title_col = "title" if "title" in df_test.columns else ("name" if "name" in df_test.columns else None)

# Fill title from publications by PMID if summaries do not carry title/name.
_pub_pmid_col = "pmid" if "pmid" in df_pubs.columns else df_pubs.columns[0]
_pub_title_col = "name" if "name" in df_pubs.columns else ("title" if "title" in df_pubs.columns else None)
pub_title_lookup = {}
if _pub_title_col is not None:
    pub_title_lookup = (
        df_pubs
        .assign(_pmid_key=lambda df: df[_pub_pmid_col].astype(str))
        .drop_duplicates("_pmid_key")
        .set_index("_pmid_key")[_pub_title_col]
        .astype(str)
        .to_dict()
    )

pubmed_latents, pubmed_pmids = load_latent("pubmed_images")
pubmed_pmids = np.asarray(pubmed_pmids).astype(str)
test_pmids = df_test[_pmid_col].astype(str).values
mask = np.isin(pubmed_pmids, test_pmids)
aligned_latents = pubmed_latents[mask]
aligned_pmids = pubmed_pmids[mask]

pmid_to_row = (
    df_test
    .assign(_pmid_key=lambda df: df[_pmid_col].astype(str))
    .drop_duplicates("_pmid_key")
    .set_index("_pmid_key")
)

pubmed_data = []
for i, pmid in enumerate(aligned_pmids):
    if pmid not in pmid_to_row.index:
        continue
    row = pmid_to_row.loc[pmid]
    title = str(row[_title_col]) if _title_col is not None and _title_col in row.index else pub_title_lookup.get(pmid, "")
    summary = str(row[_summary_col]) if _summary_col in row.index else ""
    pubmed_data.append({
        "pmid": pmid,
        "latent": aligned_latents[i],
        "short_gt": title,
        "long_gt": summary,
    })

pubmed_eval = pubmed_data[:MAX_B2T] if MAX_B2T else pubmed_data
print(f"PubMed summary test samples: {len(pubmed_eval)} / {len(pubmed_data)}")
print(f"Summaries table rows: {len(df_summaries):,}; test rows: {len(df_test):,}")

PubMed summary test samples: 2987 / 2987
Summaries table rows: 30,826; test rows: 3,945


In [34]:
df_nv = load_dataset("neurovault_text")
df_nv_meta = load_dataset("neurovault_images_meta")
nv_latents = load_latent("neurovault_images")

_doi_pub = "doi" if "doi" in df_nv.columns else df_nv.columns[0]
_doi_meta = "doi" if "doi" in df_nv_meta.columns else df_nv_meta.columns[0]
_title_nv = "title" if "title" in df_nv.columns else df_nv.columns[1]
_abs_nv = "abstract" if "abstract" in df_nv.columns else df_nv.columns[2]

neurovault_data = []
for _, pub_row in df_nv.iterrows():
    doi = pub_row[_doi_pub]
    img_indices = np.where((df_nv_meta[_doi_meta] == doi).values)[0]
    if len(img_indices) == 0 or img_indices[0] >= len(nv_latents):
        continue
    neurovault_data.append({
        "doi": doi,
        "latent": nv_latents[int(img_indices[0])],
        "short_gt": str(pub_row[_title_nv]),
        "long_gt": str(pub_row[_abs_nv]),
    })

neurovault_eval = neurovault_data[:MAX_B2T] if MAX_B2T else neurovault_data
print(f"NeuroVault samples: {len(neurovault_eval)} / {len(neurovault_data)}")

NeuroVault samples: 312 / 312


## Metric Helpers

`nvlm_sim` is a cosine similarity in NeuroVLM's shared latent space. A value around 0.33 can be meaningful because it is measured against a broad retrieval space, so the notebook plots it with empirical quartiles and a random-pair baseline rather than treating 1.0 as the only intuitive target.

In [35]:
def _semantic_sim(gen: str, gt: str) -> float:
    emb1 = _st_model.encode(gen, convert_to_tensor=True)
    emb2 = _st_model.encode(gt, convert_to_tensor=True)
    return float(st_util.cos_sim(emb1, emb2))


def _bertscore_single(generated: str, reference: str):
    p, r, f1 = bert_score(
        cands=[generated],
        refs=[reference],
        lang="en",
        model_type=BERTSCORE_MODEL,
        verbose=False,
    )
    return float(p[0]), float(r[0]), float(f1[0])


def _nvlm_latent_sim(brain_query_emb: torch.Tensor, generated: str) -> float:
    nvlm._ensure_projection_heads()
    with torch.no_grad():
        raw_emb = nvlm._encode_text(generated)
        z_text = nvlm._proj_head_text_infonce(raw_emb.to(nvlm.device))
        z_text = F.normalize(z_text, dim=-1).cpu()
    z_brain = brain_query_emb.cpu()
    if z_brain.dim() == 1:
        z_brain = z_brain.unsqueeze(0)
    return float(F.cosine_similarity(z_brain, z_text))


def _format_context_summary(table):
    lines = []
    for _, row in table.iterrows():
        lines.append(f"[{row.get('dataset', '?')}] sim={row.get('cosine_similarity', float('nan')):.3f} | {row.get('title', '')}")
    return "\n".join(lines)


def _b2t_once(table, user_prompt, gt_text, max_tokens, brain_query_emb):
    generated = nvlm.generate_llm_response(
        backend=LLM_BACKEND,
        model_name=LLM_MODEL,
        table=table,
        user_prompt=user_prompt,
        max_new_tokens=max_tokens,
        verbose=False,
    )
    bert_p, bert_r, bert_f1 = _bertscore_single(generated, gt_text)
    return {
        "generated": generated,
        "gt_text": gt_text,
        "bert_p": bert_p,
        "bert_r": bert_r,
        "bert_f1": bert_f1,
        "sem_sim": _semantic_sim(generated, gt_text),
        "nvlm_sim": _nvlm_latent_sim(brain_query_emb, generated),
    }


def run_b2t(name, latent, short_gt, long_gt, short_prompt, long_prompt="", short_tokens=64, long_tokens=512, datasets=None):
    try:
        result = nvlm.brain(latent).to_text(datasets=datasets or B2T_DATASETS)
        all_table = result.top_k(B2T_TOP_K)
        table = all_table[all_table["cosine_similarity"] > B2T_SIM_THR]
        if table.empty:
            table = all_table
        if len(table) > B2T_TOP_K:
            table = table.nlargest(B2T_TOP_K, "cosine_similarity").reset_index(drop=True)

        records = []
        for mode, prompt, gt, tokens in [
            ("short", short_prompt, short_gt, short_tokens),
            ("long", long_prompt, long_gt, long_tokens),
        ]:
            rec = _b2t_once(table, prompt, gt, tokens, result.query_embeddings)
            rec.update({"name": name, "mode": mode, "context_summary": _format_context_summary(table)})
            records.append(rec)
        return records
    except Exception as e:
        print(f"[B2T error] {name}: {type(e).__name__}: {e}")
        traceback.print_exc()
        return []


SHORT_PROMPT_GENERAL = (
    "Reply with a single concise sentence (10-20 words) naming the main cognitive "
    "function or brain network. Output only that sentence."
)
SHORT_PROMPT_PUBMED = (
    "Generate only a paper title (6-12 words) for the neuroimaging study this "
    "brain activation pattern represents. Output the title only."
)
LONG_PROMPT = ""

In [36]:
def normalize_label_text(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()


def _network_aliases(row):
    aliases = []
    for value in [row["display"], row.get("mapped_terms", ""), row.get("raw_aliases", "")]:
        aliases.extend([normalize_label_text(x) for x in str(value).split(";")])
    aliases.append(normalize_label_text(row["network_key"].replace("_", " ")))
    return [x for x in dict.fromkeys(aliases) if x]


NETWORK_LABEL_ROWS = network_info.to_dict("records")
NETWORK_ALIAS_MAP = {row["network_key"]: _network_aliases(row) for row in NETWORK_LABEL_ROWS}


def predict_network_label(text: str, min_semantic_margin=0.02):
    text_norm = normalize_label_text(text)
    alias_hits = []
    for key, aliases in NETWORK_ALIAS_MAP.items():
        for alias in aliases:
            if alias and re.search(rf"\b{re.escape(alias)}\b", text_norm):
                alias_hits.append((key, alias))
                break
    if len(alias_hits) == 1:
        return alias_hits[0][0], "alias", alias_hits[0][1], 1.0

    label_texts = [f"{row['display']}. {row['long_definition']}" for row in NETWORK_LABEL_ROWS]
    generated_emb = _st_model.encode(text, convert_to_tensor=True)
    label_emb = _st_model.encode(label_texts, convert_to_tensor=True)
    sims = st_util.cos_sim(generated_emb, label_emb).cpu().numpy().ravel()
    order = sims.argsort()[::-1]
    keys = [row["network_key"] for row in NETWORK_LABEL_ROWS]
    margin = float(sims[order[0]] - sims[order[1]]) if len(order) > 1 else float("nan")
    method = "semantic" if margin >= min_semantic_margin else "semantic_low_margin"
    best_row = NETWORK_LABEL_ROWS[order[0]]
    return keys[order[0]], method, best_row["display"], float(sims[order[0]])


def add_network_label_accuracy(df):
    out = df.copy()
    preds = out["generated"].apply(predict_network_label)
    out["pred_network_key"] = [p[0] for p in preds]
    out["label_match_method"] = [p[1] for p in preds]
    out["label_match_evidence"] = [p[2] for p in preds]
    out["label_match_score"] = [p[3] for p in preds]
    out["true_network_key"] = out["name"].map(lambda name: networks_data[name]["network_key"])
    out["network_label_correct"] = out["pred_network_key"] == out["true_network_key"]
    return out

## Retrieval Evidence Metrics

The retrieval evidence evaluation is split into two independent analyses that run before generated-text evaluation.

### MeSH Node-Type Filter

PubMed gold labels are MeSH terms, but not every MeSH category is inferable from a brain map. The default PubMed retrieval corpus excludes molecular terms. Inspect this cell and set `INCLUDE_MOLECULAR_MESH = True` if you want molecular terms included.

In [ ]:
INCLUDE_MOLECULAR_MESH = False
MESH_BRAIN_RANKABLE_NODE_TYPES = [
    "disorder",
    "anatomical_region",
    "biological_process",
    "cognitive_construct",
]
if INCLUDE_MOLECULAR_MESH:
    MESH_BRAIN_RANKABLE_NODE_TYPES.append("molecular")

PUBMED_B2T_DATASET = "kg_mesh_brain_rankable_plus_molecular" if INCLUDE_MOLECULAR_MESH else "kg_mesh_brain_rankable"
print(f"PubMed B2T will retrieve from: {PUBMED_B2T_DATASET}")
print(f"Allowed PubMed MeSH node types: {MESH_BRAIN_RANKABLE_NODE_TYPES}")

mesh_nodes = load_dataset("mesh_kg_nodes")
if "node_type" in mesh_nodes.columns:
    display(mesh_nodes["node_type"].value_counts().rename("n_terms"))
    molecular_examples = mesh_nodes[mesh_nodes["node_type"] == "molecular"].head(50)
    display(molecular_examples)
else:
    print("mesh_kg_nodes is missing node_type; PubMed node-type filtering cannot run.")

PubMed B2T will retrieve from: kg_mesh_brain_rankable_plus_molecular
Allowed PubMed MeSH node types: ['disorder', 'anatomical_region', 'biological_process', 'cognitive_construct', 'molecular']


node_type
molecular              10671
disorder                5087
organism                3938
other                   3196
method                  3023
anatomical_region       1908
biological_process      1886
cognitive_construct     1056
demographic              345
Name: n_terms, dtype: int64

,node_id,name,node_type,source
0,D000001,Calcimycin,molecular,mesh
1,D000002,Temefos,molecular,mesh
16,D000017,ABO Blood-Group System,molecular,mesh
18,D000019,Abortifacient Agents,molecular,mesh
19,D000020,"Abortifacient Agents, Nonsteroidal",molecular,mesh
20,D000021,"Abortifacient Agents, Steroidal",molecular,mesh
35,D000036,Abrin,molecular,mesh
39,D000040,Abscisic Acid,molecular,mesh
69,D000070,Acebutolol,molecular,mesh
70,D000072,Acedapsone,molecular,mesh


In [38]:
TERM_EVAL_KS = [1, 5, 10, 20]
B2T_TERM_TOP_K = max(TERM_EVAL_KS)
B2T_EVIDENCE_TOP_K = B2T_TOP_K
NETWORK_B2T_TERM_DATASETS = ["llm_neuro_terms", "kg_mesh", "cogatlas"]
B2T_TERM_DATASETS_BY_EVAL_DATASET = {
    "networks": NETWORK_B2T_TERM_DATASETS,
    "pubmed": [PUBMED_B2T_DATASET],
    "neurovault": NETWORK_B2T_TERM_DATASETS,
}
B2T_RETRIEVAL_TABLE_CACHE = {}


def normalize_term_text(text: str) -> str:
    text = str(text or "").lower()
    text = text.split("/")[0]
    return re.sub(r"[^a-z0-9]+", " ", text).strip()


def split_gold_terms(value) -> list[str]:
    if pd.isna(value):
        return []
    terms = []
    for chunk in re.split(r";|\n|\|", str(value)):
        term = chunk.strip()
        if term:
            terms.append(term)
    return terms


def terms_for_dataset(dataset_name: str) -> set[str]:
    df = load_dataset(dataset_name)
    if not isinstance(df, pd.DataFrame):
        return set()
    for col in ["term", "title", "name", "label"]:
        if col in df.columns:
            return {normalize_term_text(x) for x in df[col].dropna().astype(str)}
    return set()


def network_gold_terms(sample_name: str) -> list[str]:
    d = networks_data[sample_name]
    label_rows = network_labels_df[network_labels_df["raw_network_label"].astype(str) == str(d["raw_network_label"])]
    if label_rows.empty:
        label_rows = network_labels_df[network_labels_df["network_key"] == d["network_key"]]
    terms = []
    for col in ["mapped_terms", "region_terms", "cognitive_terms"]:
        if col in label_rows.columns:
            for value in label_rows[col].dropna().tolist():
                terms.extend(split_gold_terms(value))
    return list(dict.fromkeys(terms))


def load_pubmed_mesh_annotations_or_none():
    try:
        annotations = load_dataset("pubmed_mesh_annotations")
        print(f"Loaded PubMed MeSH annotations for {len(annotations):,} PMIDs")
        return annotations
    except Exception as e:
        print("PubMed MeSH gold annotations are unavailable; PubMed exact term-ranking metrics will be skipped.")
        print("Upload mesh_annotations.json to neurovlm/mesh_kg to enable them.")
        print(f"Loader error: {type(e).__name__}: {e}")
        return None


PUBMED_MESH_ANNOTATIONS = load_pubmed_mesh_annotations_or_none()
MESH_NODE_TYPE_BY_TERM = {}
if "node_type" in mesh_nodes.columns:
    name_col = "name" if "name" in mesh_nodes.columns else "term"
    MESH_NODE_TYPE_BY_TERM = {
        normalize_term_text(row[name_col]): row["node_type"]
        for _, row in mesh_nodes.iterrows()
        if pd.notna(row.get(name_col)) and pd.notna(row.get("node_type"))
    }


def pubmed_gold_terms(pmid) -> list[str]:
    if PUBMED_MESH_ANNOTATIONS is None:
        return []
    raw_terms = PUBMED_MESH_ANNOTATIONS.get(str(pmid), [])
    filtered = []
    for term in raw_terms:
        base = str(term).split("/")[0].strip()
        norm = normalize_term_text(base)
        if norm and MESH_NODE_TYPE_BY_TERM.get(norm) in set(MESH_BRAIN_RANKABLE_NODE_TYPES):
            filtered.append(base)
    return list(dict.fromkeys(filtered))


def table_terms(table: pd.DataFrame) -> list[str]:
    if table is None or table.empty:
        return []
    return [str(x) for x in table["title"].fillna("").tolist() if str(x).strip()]


def retrieval_table_for_sample(latent, dataset_name: str, sample: str):
    cache_key = (dataset_name, str(sample))
    if cache_key in B2T_RETRIEVAL_TABLE_CACHE:
        return B2T_RETRIEVAL_TABLE_CACHE[cache_key]
    datasets = B2T_TERM_DATASETS_BY_EVAL_DATASET[dataset_name]
    result = nvlm.brain(latent).to_text(datasets=datasets)
    all_table = result.top_k(max(B2T_TERM_TOP_K, B2T_EVIDENCE_TOP_K))
    table = all_table[all_table["cosine_similarity"] > B2T_SIM_THR]
    if table.empty:
        table = all_table
    table = table.nlargest(max(B2T_TERM_TOP_K, B2T_EVIDENCE_TOP_K), "cosine_similarity").reset_index(drop=True)
    B2T_RETRIEVAL_TABLE_CACHE[cache_key] = table
    return table


_pub_abs_col = "description" if "description" in df_pubs.columns else ("abstract" if "abstract" in df_pubs.columns else None)
_pub_abs_lookup = {}
if _pub_abs_col is not None:
    _pub_abs_lookup = (
        df_pubs
        .assign(_pmid_key=lambda df: df[_pub_pmid_col].astype(str))
        .drop_duplicates("_pmid_key")
        .set_index("_pmid_key")[_pub_abs_col]
        .astype(str)
        .to_dict()
    )


def dataset_records_for_retrieval_eval():
    if RUN_NETWORKS:
        for name, d in networks_data.items():
            yield "networks", name, d["latent"], network_gold_terms(name), {"long_description": d["long_gt"]}
    if RUN_PUBMED:
        for d in pubmed_eval:
            pmid = str(d["pmid"])
            references = {"summary": d["long_gt"]}
            abstract = _pub_abs_lookup.get(pmid, "")
            if abstract:
                references["abstract"] = abstract
            yield "pubmed", pmid, d["latent"], pubmed_gold_terms(pmid), references
    if RUN_NEUROVAULT:
        for d in neurovault_eval:
            yield "neurovault", str(d["doi"]), d["latent"], [], {"abstract": d["long_gt"]}

Loaded PubMed MeSH annotations for 1,231,613 PMIDs


### Approach 1: Gold-Term Ranking

This evaluates whether the ranked terms retrieved from the brain include known gold terms. It runs for networks and PubMed only; NeuroVault is skipped because we do not have gold terms for it.

In [39]:
def exact_term_ranking_rows(dataset: str, sample: str, gold_terms: list[str], retrieved_terms: list[str], reachable_terms: set[str] | None = None):
    gold_norm_all = {normalize_term_text(t) for t in gold_terms if normalize_term_text(t)}
    if reachable_terms is None:
        gold_norm = gold_norm_all
        excluded = set()
    else:
        gold_norm = gold_norm_all & reachable_terms
        excluded = gold_norm_all - reachable_terms
    retrieved_norm = [normalize_term_text(t) for t in retrieved_terms if normalize_term_text(t)]
    rows = []
    if not gold_norm:
        return rows
    first_hit_rank = next((i + 1 for i, term in enumerate(retrieved_norm) if term in gold_norm), np.nan)
    for k in TERM_EVAL_KS:
        topk = retrieved_norm[:k]
        hits = set(topk) & gold_norm
        rows.append({
            "dataset": dataset,
            "sample": sample,
            "k": k,
            "n_gold_terms": len(gold_norm),
            "n_unreachable_gold_terms": len(excluded),
            "n_retrieved_terms": len(topk),
            "n_hits": len(hits),
            "precision_at_k": len(hits) / max(len(topk), 1),
            "recall_at_k": len(hits) / len(gold_norm),
            "hit_at_k": bool(hits),
            "mrr": 0.0 if pd.isna(first_hit_rank) or first_hit_rank > k else 1.0 / float(first_hit_rank),
            "matched_terms": "; ".join(sorted(hits)),
        })
    return rows


network_candidate_terms = set().union(*(terms_for_dataset(ds) for ds in NETWORK_B2T_TERM_DATASETS))
network_missing_rows = []
for sample_name in networks_data:
    for term in network_gold_terms(sample_name):
        norm = normalize_term_text(term)
        if norm and norm not in network_candidate_terms:
            network_missing_rows.append({"sample": sample_name, "term": term, "normalized_term": norm})
network_missing_terms_df = pd.DataFrame(network_missing_rows).drop_duplicates() if network_missing_rows else pd.DataFrame(columns=["sample", "term", "normalized_term"])
display(network_missing_terms_df.head(25))
print(f"Network gold terms missing from llm_neuro_terms + kg_mesh + cogatlas: {len(network_missing_terms_df):,}")
if len(network_missing_terms_df):
    network_missing_terms_df.to_csv(OUTPUT_DIR / "network_gold_terms_missing_from_retrieval_corpora.csv", index=False)

pubmed_candidate_terms = terms_for_dataset(PUBMED_B2T_DATASET)

,sample,term,normalized_term


Network gold terms missing from llm_neuro_terms + kg_mesh + cogatlas: 0


In [40]:
term_metric_rows = []
term_examples = []

for dataset, sample, latent, gold_terms, _references in tqdm(list(dataset_records_for_retrieval_eval()), desc="Approach 1 gold-term ranking"):
    if dataset == "neurovault":
        continue
    table = retrieval_table_for_sample(latent, dataset, sample)
    retrieved_terms = table_terms(table)
    reachable_terms = network_candidate_terms if dataset == "networks" else pubmed_candidate_terms
    term_metric_rows.extend(exact_term_ranking_rows(dataset, sample, gold_terms, retrieved_terms, reachable_terms=reachable_terms))
    term_examples.append({
        "dataset": dataset,
        "sample": sample,
        "gold_terms": "; ".join(gold_terms[:50]),
        "top_terms": "; ".join(retrieved_terms[:B2T_TERM_TOP_K]),
    })

b2t_term_metrics_df = pd.DataFrame(term_metric_rows)
b2t_term_examples_df = pd.DataFrame(term_examples)
b2t_term_metrics_df.to_csv(OUTPUT_DIR / "b2t_approach1_gold_term_ranking_metrics.csv", index=False)
b2t_term_examples_df.to_csv(OUTPUT_DIR / "b2t_approach1_gold_term_examples.csv", index=False)

if len(b2t_term_metrics_df):
    display(b2t_term_metrics_df.groupby(["dataset", "k"])[["precision_at_k", "recall_at_k", "hit_at_k", "mrr", "n_unreachable_gold_terms"]].mean().round(3))
else:
    print("No exact gold-term metrics were computed. PubMed needs mesh_annotations.json; NeuroVault has no gold terms.")

Approach 1 gold-term ranking:   0%|          | 0/3418 [00:00<?, ?it/s]

precision_at_k  recall_at_k  hit_at_k    mrr  \
dataset  k                                                  
networks 1            0.109        0.005     0.109  0.109   
         5            0.144        0.031     0.345  0.198   
         10           0.136        0.057     0.420  0.209   
         20           0.127        0.104     0.563  0.218   
pubmed   1            0.035        0.006     0.035  0.035   
         5            0.026        0.018     0.097  0.056   
         10           0.023        0.027     0.136  0.061   
         20           0.021        0.040     0.182  0.065   

             n_unreachable_gold_terms  
dataset  k                             
networks 1                        0.0  
         5                        0.0  
         10                       0.0  
         20                       0.0  
pubmed   1                        0.0  
         5                        0.0  
         10                       0.0  
         20                       0.0

### Approach 3: Ground-Truth Text vs Retrieved Evidence

This evaluates whether the top retrieved terms and definitions collectively resemble the ground-truth text. It runs for networks, PubMed, and NeuroVault.

In [ ]:
def evidence_text(table: pd.DataFrame, k: int = B2T_EVIDENCE_TOP_K) -> str:
    lines = []
    for _, row in table.head(k).iterrows():
        title = str(row.get("title", "")).strip()
        desc = str(row.get("description", "")).strip()
        if title or desc:
            lines.append(f"{title}. {desc}".strip())
    return "\n".join(lines)


def sentence_cosine(a: str, b: str) -> float:
    if not str(a or "").strip() or not str(b or "").strip():
        return np.nan
    emb = _st_model.encode([a, b], convert_to_tensor=True, normalize_embeddings=True)
    return float(st_util.cos_sim(emb[0], emb[1]))


_specter_metric_model = None


def get_specter_metric_model():
    global _specter_metric_model
    if _specter_metric_model is None:
        from neurovlm.models import Specter
        device = "cuda" if torch.cuda.is_available() else "cpu"
        _specter_metric_model = Specter(
            "allenai/specter2_aug2023refresh",
            adapter="adhoc_query",
            device=device,
        )
    return _specter_metric_model


def specter_cosine(a: str, b: str) -> float:
    if not str(a or "").strip() or not str(b or "").strip():
        return np.nan
    specter = get_specter_metric_model()
    with torch.no_grad():
        emb = specter([
            {"title": str(a), "summary": ""},
            {"title": str(b), "summary": ""},
        ]).detach().cpu()
        emb = F.normalize(emb, dim=-1)
    return float(F.cosine_similarity(emb[0:1], emb[1:2]).item())


def evidence_similarity_rows(dataset: str, sample: str, table: pd.DataFrame, references: dict[str, str]):
    evidence = evidence_text(table)
    rows = []
    for ref_name, ref_text in references.items():
        rows.append({
            "dataset": dataset,
            "sample": sample,
            "reference": ref_name,
            "evidence_top_k": B2T_EVIDENCE_TOP_K,
            "minilm_evidence_similarity": sentence_cosine(ref_text, evidence),
            "specter_evidence_similarity": specter_cosine(ref_text, evidence),
            "reference_text": str(ref_text)[:500],
            "retrieval_evidence": evidence[:1000],
        })
    return rows


In [ ]:
evidence_metric_rows = []
retrieval_examples = []

for dataset, sample, latent, gold_terms, references in tqdm(list(dataset_records_for_retrieval_eval()), desc="Approach 3 evidence similarity"):
    table = retrieval_table_for_sample(latent, dataset, sample)
    evidence_metric_rows.extend(evidence_similarity_rows(dataset, sample, table, references))
    retrieval_examples.append({
        "dataset": dataset,
        "sample": sample,
        "gold_terms": "; ".join(gold_terms[:50]),
        "top_terms": "; ".join(table_terms(table)[:B2T_TERM_TOP_K]),
        "top_context": evidence_text(table),
    })

b2t_evidence_similarity_df = pd.DataFrame(evidence_metric_rows)
b2t_retrieval_examples_df = pd.DataFrame(retrieval_examples)
b2t_evidence_similarity_df.to_csv(OUTPUT_DIR / "b2t_approach3_retrieval_evidence_similarity.csv", index=False)
b2t_retrieval_examples_df.to_csv(OUTPUT_DIR / "b2t_approach3_retrieval_evidence_examples.csv", index=False)

display(b2t_evidence_similarity_df.groupby(["dataset", "reference"])[["minilm_evidence_similarity", "specter_evidence_similarity"]].agg(["mean", "std", "count"]).round(3))

In [43]:
if RUN_GENERATED_TEXT:
    # Pre-download Hugging Face assets used by BERTScore so metric calls do not
    # repeatedly block on tokenizer/model metadata requests during the eval loop.
    def predownload_hf_model(model_name: str):
        print(f"Pre-downloading Hugging Face model for BERTScore: {model_name}")
        AutoTokenizer.from_pretrained(model_name, use_fast=False)
        AutoModel.from_pretrained(model_name)
        print("BERTScore model is cached.")

    predownload_hf_model(BERTSCORE_MODEL)
else:
    print("Skipping BERTScore model pre-download because RUN_GENERATED_TEXT = False.")


Skipping BERTScore model pre-download because RUN_GENERATED_TEXT = False.


## Generated Text Evaluation

This slow LLM/BERTScore section is disabled by default. Set `RUN_GENERATED_TEXT = True` to run generated text vs. ground-truth text metrics.


In [44]:
if RUN_GENERATED_TEXT:
    b2t_frames = []

    if RUN_NETWORKS:
        records = []
        for net_name, d in tqdm(networks_data.items(), desc="Networks B2T"):
            records.extend(run_b2t(net_name, d["latent"], d["short_gt"], d["long_gt"], SHORT_PROMPT_GENERAL, LONG_PROMPT))
        b2t_net_df = add_network_label_accuracy(pd.DataFrame(records))
        b2t_net_df["dataset"] = "networks"
        b2t_frames.append(b2t_net_df)

    if RUN_PUBMED:
        records = []
        for d in tqdm(pubmed_eval, desc="PubMed B2T"):
            records.extend(run_b2t(str(d["pmid"]), d["latent"], d["short_gt"], d["long_gt"], SHORT_PROMPT_PUBMED, LONG_PROMPT, datasets=[PUBMED_B2T_DATASET]))
        b2t_pubmed_df = pd.DataFrame(records)
        b2t_pubmed_df["dataset"] = "pubmed"
        b2t_frames.append(b2t_pubmed_df)

    if RUN_NEUROVAULT:
        records = []
        for d in tqdm(neurovault_eval, desc="NeuroVault B2T"):
            records.extend(run_b2t(str(d["doi"]), d["latent"], d["short_gt"], d["long_gt"], SHORT_PROMPT_GENERAL, LONG_PROMPT))
        b2t_nv_df = pd.DataFrame(records)
        b2t_nv_df["dataset"] = "neurovault"
        b2t_frames.append(b2t_nv_df)

    b2t_all = pd.concat(b2t_frames, ignore_index=True)
    b2t_all.to_csv(OUTPUT_DIR / "brain_to_text_metrics_v2.csv", index=False)
    b2t_all.head()
else:
    print("Skipping generated text evaluation because RUN_GENERATED_TEXT = False.")
    b2t_all = pd.DataFrame()


Skipping generated text evaluation because RUN_GENERATED_TEXT = False.


In [45]:
if RUN_GENERATED_TEXT and len(b2t_all):
    def recall_at_1(df: pd.DataFrame) -> float:
        if len(df) < 2:
            return np.nan
        generated = df["generated"].astype(str).tolist()
        with torch.no_grad():
            raw = nvlm._encode_text(generated)
            z_text = nvlm._proj_head_text_infonce(raw.to(nvlm.device))
            z_text = F.normalize(z_text, dim=-1).cpu()
        brain_embs = []
        for _, row in df.iterrows():
            source = row["dataset"]
            name = row["name"]
            if source == "networks":
                brain_embs.append(networks_data[name]["latent"])
            elif source == "pubmed":
                brain_embs.append(next(d["latent"] for d in pubmed_eval if str(d["pmid"]) == str(name)))
            elif source == "neurovault":
                brain_embs.append(next(d["latent"] for d in neurovault_eval if str(d["doi"]) == str(name)))
        z_brain = F.normalize(torch.stack([b.cpu() for b in brain_embs]), dim=-1)
        scores = z_text @ z_brain.T
        return float((scores.argmax(dim=1) == torch.arange(len(df))).float().mean())

    recall_rows = []
    for (dataset, mode), sub in b2t_all.groupby(["dataset", "mode"]):
        recall_rows.append({"dataset": dataset, "mode": mode, "recall_at_1": recall_at_1(sub.reset_index(drop=True)), "n": len(sub)})
    recall_df = pd.DataFrame(recall_rows)
    summary = b2t_all.groupby(["dataset", "mode"])[["nvlm_sim", "bert_f1", "sem_sim"]].agg(["mean", "std", "count"]).round(3)
    display(summary)
    display(recall_df.round(3))

    if "network_label_correct" in b2t_all.columns:
        label_summary = b2t_all[b2t_all["dataset"] == "networks"].groupby("mode")["network_label_correct"].agg(["mean", "sum", "count"]).round(3)
        display(label_summary)
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.


In [46]:
if RUN_GENERATED_TEXT and len(b2t_all):
    b2t_all[(b2t_all["mode"] == "short") & (b2t_all["dataset"] == "pubmed")].iloc[0].get("generated")
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.


In [47]:
if RUN_GENERATED_TEXT and len(b2t_all):
    '''
    '[kg_mesh] sim=0.534 | surgical equipment\n
     [kg_mesh] sim=0.529 | acetylene\n
     [kg_mesh] sim=0.522 | equipment and supplies, hospital\n
     [kg_mesh] sim=0.522 | hospitals, packaged\n
     [kg_mesh] sim=0.519 | cellular neural networks, computer'
    '''
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.


In [48]:
if RUN_GENERATED_TEXT and len(b2t_all):
    ''''
    Acoustic neuromas, benign tumors of the auditory nerve, exhibit correlations between their morphology and 
    otoneurological manifestations. The size and site of origin of these tumors influence clinical presentation,
     with patients having lateral neuromas typically experiencing early subjective hearing loss due to smaller 
     tumors often confined to the internal auditory canal. In contrast, medial neuromas tend to be larger and 
     may grow without causing significant audiological symptoms, often preserving normal hearing function. 
     The sensitivity of the stapedial reflex test is higher for lateral neuromas, while vestibular tests 
     indicate a higher frequency of central vestibular involvement in larger tumors. The combination of 
     brainstem auditory evoked potentials and vestibular tests can effectively identify acoustic neuromas,
       offering an optimal level of sensitivity.'


    '''
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.


In [49]:
if RUN_GENERATED_TEXT and len(b2t_all):
    '''
    'Surgical Equipment encompasses a range of instruments and tools essential for medical interventions, including those related to Magnetic Resonance Imaging (MRI). 
    Acetylene, though not directly involved in biological processes, plays a role in MRI by being used in the cooling systems of some MRI machines. 
    Equipment and Supplies, Hospital, refers to the diverse set of medical devices and consumables utilized in hospitals for various diagnostic procedures, 
    which can include MRI and other imaging techniques. Hospitals, Packaged, denotes pre-assembled medical equipment and supplies designed for efficient use in 
    clinical settings, often including items that support MRI and ultrasonography. These components collectively ensure the smooth operation and precision of surgical 
    and diagnostic procedures, highlighting the integral role of specialized tools and technology in modern medical practice.'
    '''
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.


In [50]:
if RUN_GENERATED_TEXT and len(b2t_all):
    '''
    "Neuroimaging Evidence of Surgical Equipment Usage in MRI Procedures\n\nSurgical equipment and supplies used in hospitals are crucial for various medical 
    interventions, including those related to Magnetic Resonance Imaging (MRI). Acetylene, though not a direct component of brain activity during MRI, plays 
    a role in the technology's operation. Hospitals often package these surgical tools and supplies together, facilitating efficient use in diagnostic procedures. 
    The underlying neural network supporting such tasks might involve cellular neural networks, which model biological neurons to recognize patterns and process signals,
    reflecting the complex integration of medical equipment with brain functions during imaging procedures."
    '''
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.


In [51]:
if RUN_GENERATED_TEXT and len(b2t_all):
    "'Acoustic neuroma: correlations between morphology and otoneurological manifestations.'"
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.


In [52]:
if RUN_GENERATED_TEXT and len(b2t_all):
    plot_df = b2t_all.copy()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    metric_specs = [("nvlm_sim", "NeuroVLM latent similarity"), ("bert_f1", "BERTScore F1"), ("sem_sim", "Sentence semantic similarity")]
    for ax, (metric, title) in zip(axes, metric_specs):
        groups = [g[metric].dropna().values for _, g in plot_df.groupby("dataset")]
        labels = [k for k, _ in plot_df.groupby("dataset")]
        ax.boxplot(groups, labels=labels, showmeans=True)
        ax.set_title(title)
        ax.set_ylim(min(-0.05, np.nanmin(plot_df[metric]) - 0.05), min(1.05, np.nanmax(plot_df[metric]) + 0.1))
        ax.grid(axis="y", alpha=0.25)
        ax.tick_params(axis="x", rotation=15)
    fig.tight_layout()
    plt.savefig(OUTPUT_DIR / "b2t_metric_distributions.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.


In [53]:
if RUN_GENERATED_TEXT and len(b2t_all):
    def _brain_latents_for_group(df):
        brain_embs = []
        for _, row in df.iterrows():
            source = row["dataset"]
            name = row["name"]
            if source == "networks":
                brain_embs.append(networks_data[name]["latent"])
            elif source == "pubmed":
                brain_embs.append(next(d["latent"] for d in pubmed_eval if str(d["pmid"]) == str(name)))
            elif source == "neurovault":
                brain_embs.append(next(d["latent"] for d in neurovault_eval if str(d["doi"]) == str(name)))
        return F.normalize(torch.stack([b.cpu() for b in brain_embs]), dim=-1)


    def _text_latents_for_group(df):
        nvlm._ensure_projection_heads()
        generated = df["generated"].astype(str).tolist()
        with torch.no_grad():
            raw = nvlm._encode_text(generated)
            z_text = nvlm._proj_head_text_infonce(raw.to(nvlm.device))
            return F.normalize(z_text, dim=-1).cpu()

    baseline_rows = []
    for (dataset, mode), sub in b2t_all.groupby(["dataset", "mode"]):
        if len(sub) < 2:
            continue
        sub = sub.reset_index(drop=True)
        z_text = _text_latents_for_group(sub)
        z_brain = _brain_latents_for_group(sub)
        scores = z_text @ z_brain.T
        eye = torch.eye(len(sub), dtype=torch.bool)
        for val in scores[eye].numpy():
            baseline_rows.append({"dataset": dataset, "mode": mode, "pair": "matched", "score": float(val)})
        for val in scores[~eye].numpy():
            baseline_rows.append({"dataset": dataset, "mode": mode, "pair": "random/off-diagonal", "score": float(val)})

    baseline_df = pd.DataFrame(baseline_rows)
    fig, ax = plt.subplots(figsize=(8, 4))
    if len(baseline_df):
        matched = baseline_df[baseline_df["pair"] == "matched"]["score"]
        random_pairs = baseline_df[baseline_df["pair"] == "random/off-diagonal"]["score"]
        bins = np.linspace(min(baseline_df["score"].min(), 0), baseline_df["score"].max(), 24)
        ax.hist(random_pairs, bins=bins, alpha=0.55, label="random/off-diagonal pairs", color="lightgray", edgecolor="white")
        ax.hist(matched, bins=bins, alpha=0.75, label="matched generated text", color="steelblue", edgecolor="white")
        for x, lab, color in [(matched.median(), "matched median", "steelblue"), (random_pairs.median(), "random median", "gray")]:
            ax.axvline(x, color=color, linestyle="--", linewidth=1)
            ax.text(x, ax.get_ylim()[1] * 0.92, f"{lab}\n{x:.3f}", ha="center", va="top", fontsize=8)
    else:
        vals = b2t_all["nvlm_sim"].dropna()
        ax.hist(vals, bins=min(20, max(5, len(vals) // 2)), alpha=0.75, color="steelblue", edgecolor="white")
        ax.text(0.5, 0.9, "Need at least two samples per group for random-pair baseline", transform=ax.transAxes, ha="center")
    ax.set_title("How to read nvlm_sim: matched text versus random pairs")
    ax.set_xlabel("NeuroVLM latent cosine similarity")
    ax.set_ylabel("Pairs")
    ax.legend(frameon=False)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    plt.savefig(OUTPUT_DIR / "b2t_nvlm_sim_scale.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.


In [54]:
if RUN_GENERATED_TEXT and len(b2t_all):
    if "network_label_correct" in b2t_all.columns:
        net = b2t_all[b2t_all["dataset"] == "networks"].copy()
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        acc = net.groupby("mode")["network_label_correct"].mean()
        axes[0].bar(acc.index, acc.values, color=["#4c78a8", "#72b7b2"])
        axes[0].set_ylim(0, 1)
        axes[0].set_ylabel("Accuracy")
        axes[0].set_title("Network label accuracy")
        axes[0].grid(axis="y", alpha=0.25)

        cm = pd.crosstab(net["true_network_key"], net["pred_network_key"], normalize="index")
        im = axes[1].imshow(cm.values, vmin=0, vmax=1, cmap="Blues")
        axes[1].set_xticks(range(len(cm.columns)), cm.columns, rotation=45, ha="right")
        axes[1].set_yticks(range(len(cm.index)), cm.index)
        axes[1].set_title("Predicted network label by true label")
        fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        fig.tight_layout()
        plt.savefig(OUTPUT_DIR / "b2t_network_label_accuracy.png", dpi=150, bbox_inches="tight")
        plt.show()

        display(net[["name", "mode", "true_network_key", "pred_network_key", "network_label_correct", "label_match_method", "generated"]])
else:
    print("Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.")


Skipping generated-text downstream analysis because RUN_GENERATED_TEXT = False or b2t_all is empty.
